In [ ]:
# supervised.ipynb
# Intent Classification Experiments

import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report


In [ ]:
##Load data & labels
df = pd.read_pickle("data/pages_df.pkl")
# assume you have a `label` column in df:
X, y = df["text"], df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)


In [ ]:
##TF–IDF + Logistic Regression
pipe_tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(max_df=0.8, min_df=2, ngram_range=(1,2))),
    ("clf", LogisticRegression(max_iter=1000))
])
pipe_tfidf_lr.fit(X_train, y_train)
print(classification_report(y_test, pipe_tfidf_lr.predict(X_test)))

In [ ]:
##TF–IDF + SVM
pipe_tfidf_svm = Pipeline([
    ("tfidf", TfidfVectorizer(max_df=0.8, min_df=2, ngram_range=(1,2))),
    ("clf", SVC(kernel="linear"))
])
pipe_tfidf_svm.fit(X_train, y_train)
print(classification_report(y_test, pipe_tfidf_svm.predict(X_test)))

In [ ]:
##Embedding + Logistic Regression
embedder = SentenceTransformer("paraphrase-MiniLM-L6-v2")
X_train_emb = embedder.encode(X_train.tolist())
X_test_emb  = embedder.encode(X_test.tolist())

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_emb, y_train)
print(classification_report(y_test, lr.predict(X_test_emb)))

In [ ]:
##Hybrid Inference Function
def hybrid_answer(query):
    intent = pipe_tfidf_lr.predict([query])[0]       # or best classifier
    snippet = embed_retrieve(query, top_k=1)["text"].iloc[0]
    return intent, snippet

hybrid_answer("how do i reset my password?")

In [ ]:
##Save best model
pickle.dump(pipe_tfidf_lr, open("models/best_intent_clf.pkl","wb"))